In [1]:
from helper_utils import load_chroma, word_wrap, project_embeddings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import numpy as np

In [2]:
embedding_function = SentenceTransformerEmbeddingFunction()

chroma_collection = load_chroma(filename='sample.pdf', collection_name='sample', embedding_function=embedding_function)
chroma_collection.count()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

7

In [3]:
query = "How are embeddings stored in ChromaDB?"
results = chroma_collection.query(query_texts=query, n_results=10, include=['documents', 'embeddings'])

retrieved_documents = results['documents'][0]

for document in results['documents'][0]:
    print(word_wrap(document))
    print('')

mathematically similar to the query vector, effectively finding content
that is semantically related rather than just keyword - matched. this
makes chromadb far more powerful than traditional databases for tasks
involving natural language understanding, recommendation systems,
semantic search, and context retrieval in ai - driven applications. how
chromadb works the fundamental building block of chromadb is the
collection, which is roughly equivalent to a table in a traditional
relational database. each collection holds a set of documents, their
corresponding vector embeddings, and any metadata the developer wishes
to attach. when data is added to a collection, chromadb can generate
the embeddings automatically using a built - in embedding function, or
it can accept pre - computed embeddings from an external model. this
flexibility allows developers to work with any embedding model they
prefer, whether that is openai ' s text - embedding models, sentence

transformers from hugging face

In [4]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [5]:
pairs = [[query, doc] for doc in retrieved_documents]
scores = cross_encoder.predict(pairs)
print("Scores:")
for score in scores:
    print(score)

Scores:
4.3915625
1.1536274
5.67968
1.6572206
4.9204917
-2.8119597
-3.4508395


In [6]:
print("New Ordering:")
for o in np.argsort(scores)[::-1]:
    print(o+1)

New Ordering:
3
5
1
4
2
6
7


In [7]:
original_query = "How are embeddings stored in ChromaDB?"

generated_queries = [
    "What is retrieval augmented generation?",
    "How does ChromaDB retrieve relevant documents?",
    "How are embeddings used in ChromaDB?",
    "Why is ChromaDB suitable for RAG applications?",
    "What role do vector databases play in retrieval augmented generation?"
]

In [8]:
queries = [original_query] + generated_queries

results = chroma_collection.query(query_texts=queries, n_results=10, include=['documents', 'embeddings'])
retrieved_documents = results['documents']

In [9]:
# Deduplicate the retrieved documents
unique_documents = set()
for documents in retrieved_documents:
    for document in documents:
        unique_documents.add(document)

unique_documents = list(unique_documents)

In [10]:
pairs = []
for doc in unique_documents:
    pairs.append([original_query, doc])

In [11]:
scores = cross_encoder.predict(pairs)


In [12]:
print("Scores:")
for score in scores:
    print(score)

Scores:
1.6572206
-2.8119597
4.3915625
-3.4508395
5.67968
1.1536274
4.9204917


In [13]:
print("New Ordering:")
for o in np.argsort(scores)[::-1]:
    print(o)

New Ordering:
4
6
2
0
5
1
3
